In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data

In [4]:
from transformers import AutoTokenizer, AutoModel

In [4]:
class ModelConfig():
    def __init__(self):
        pass

In [5]:
## Parameters for encoder/decoder blocks and layers

d_model = 512 # dimension of inputs to enc/dec layers
n_heads = 8 # number of attention heads
n_layers = 6 # number of enc/dec layers
latent_dim = 2 # dimension of latent variable space

In [2]:
from huggingface_hub import login
login()

In [5]:
# Get tokenizer and embedding layer from pretrained model
MODEL_NAME = "meta-llama/Llama-3.1-8B" # Set in config

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Set pad tokens to eos_tokens
llama_model = AutoModel.from_pretrained(MODEL_NAME)

llama_embed = llama_model.get_input_embeddings() # becomes our nn.Embedding
dmodel = llama_embed.embedding_dim

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B.
403 Client Error. (Request ID: Root=1-69c3eb31-170be3e03c905645638a85c3;11a65df2-bb4b-4af2-90be-35935a8b5776)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.1-8B is awaiting a review from the repo authors.

In [ ]:
# How to use nested tensors and get rid of warning
class VAETransformer(nn.Module):
    def __init__(self, 
                 llama_embed,
                 dmodel,
                 nheads, 
                 nlayers, 
                 content_latent_dim, 
                 style_latent_dim,
                 pad_token_id,
                 eos_token_id
                 ):
        super().__init__()

        self.embedding = llama_embed
        self.dmodel = llama_embed.embedding_dim
        vocab_size = llama_embed.num_embeddings
        self.pad_token_id = pad_token_id
        self.eos_token_id = eos_token_id


        # Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=dmodel, 
                                                   nhead = nheads,
                                                   batch_first=False)        
        self.encoder =  nn.TransformerEncoder(encoder_layer, nlayers)

        # Decoder
        decoder_layer = nn.TransformerDecoderLayer(d_model=dmodel, 
                                                   nhead=nheads,
                                                   batch_first=False)
        self.decoder = nn.TransformerDecoder(decoder_layer, nlayers)

        # Content latent embedding
        self.content_mu = nn.Linear(dmodel, content_latent_dim)
        self.content_log_var  = nn.Linear(dmodel, content_latent_dim)

        # Style latent embedding
        self.style_mu = nn.Linear(dmodel, style_latent_dim)
        self.style_log_var = nn.Linear(dmodel, style_latent_dim)

        self.latent_to_decode = nn.Linear(
            content_latent_dim + style_latent_dim, 
            dmodel
        )

        # Output projection
        self.output_proj = nn.Linear(dmodel, vocab_size)

    def reparameterization(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    
    def encode(self, input_ids, attention_mask):
        """
        input_ids:      (batch, seq)
        attention_mask: (batch, seq), 1 = real, 0 = pad
        """
        # Convert to (seq, batch)
        x = input_ids.transpose(0, 1)
        attn = attention_mask.transpose(0, 1)

        # Embed tokens
        # Normalize embeddings by sqrt(dmodel) ala Transformer
        emb = self.embedding(x) * (self.dmodel ** 0.5) 

        # Transformer expects: True = pad
        src_key_padding_mask = (attention_mask == 0)  # (batch, seq)

        # Encode
        memory = self.encoder(
            emb,
            src_key_padding_mask=src_key_padding_mask
        )  # (seq, batch, dmodel)

        # Masked mean pooling over seq dimension
        mask_f = attn.unsqueeze(-1).float()  # (seq, batch, 1)
        summed = (memory * mask_f).sum(dim=0)  # (batch, dmodel)
        counts = mask_f.sum(dim=0).clamp_min(1.0)
        pooled = summed / counts

        # Latent parameters
        c_mu = self.content_mu(pooled)
        c_logvar = self.content_logvar(pooled)
        s_mu = self.style_mu(pooled)
        s_logvar = self.style_logvar(pooled)

        return memory, src_key_padding_mask, (c_mu, c_logvar), (s_mu, s_logvar)
    
    def _build_causal_mask(self, tgt_len, device):
        # (tgt_len, tgt_len), True = block, False = allow
        mask = torch.triu(torch.ones(tgt_len, tgt_len, device=device), diagonal=1).bool()
        return mask
    
    def decode_train(self, memory, src_key_padding_mask, z_c, z_s, target_ids):
        """
        memory: (seq_src, batch, dmodel)
        src_key_padding_mask: (batch, seq_src)
        z_c, z_s: (batch, latent_dim)
        target_ids: (batch, tgt_seq)  -- full target sequence including BOS, tokens, EOS
        """
        device = memory.device
        batch_size, tgt_seq = target_ids.shape

        # Shift targets for teacher forcing:
        # input:  [BOS, y_1, ..., y_{T-1}]
        # output: [y_1, ..., y_{T-1}, y_T]
        tgt_input = target_ids[:, :-1]   # (batch, tgt_seq-1)
        tgt_output = target_ids[:, 1:]   # (batch, tgt_seq-1)

        # Embed target tokens
        tgt_input_t = tgt_input.transpose(0, 1)  # (tgt_len, batch)
        tgt_emb = self.embedding(tgt_input_t) * (self.dmodel ** 0.5)  # (tgt_len, batch, dmodel)

        # Latent → decoder space
        z = torch.cat([z_c, z_s], dim=-1)          # (batch, latent_dim_total)
        z_dec = self.latent_to_decode(z)           # (batch, dmodel)

        # Add latent as a global bias to each time step
        z_dec_expanded = z_dec.unsqueeze(0)        # (1, batch, dmodel)
        tgt = tgt_emb + z_dec_expanded             # (tgt_len, batch, dmodel)

        tgt_len = tgt.size(0)
        causal_mask = self._build_causal_mask(tgt_len, device)  # (tgt_len, tgt_len)

        # Target padding mask (for decoder self-attention)
        tgt_key_padding_mask = (tgt_input == self.pad_token_id)  # (batch, tgt_len)
        # NEed to make pad_token_id a class attribute

        out = self.decoder(
            tgt,
            memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )  # (tgt_len, batch, dmodel)

        logits = self.output_proj(out)  # (tgt_len, batch, vocab)

        return logits, tgt_output

    @torch.no_grad()
    def decode_generate(self, memory, src_key_padding_mask, z_c, z_s,
                        max_len=64, temperature=1.0, top_k=None):
        """
        Autoregressive generation:
        returns a single sequence of token ids per batch element.
        """
        device = memory.device
        batch_size = memory.size(1)

        z = torch.cat([z_c, z_s], dim=-1)
        z_dec = self.latent_to_decode(z)  # (batch, dmodel)

        # Start with BOS for each sequence
        cur_tokens = torch.full(
            (batch_size, 1),
            self.bos_token_id,
            dtype=torch.long,
            device=device
        )  # (batch, 1)

        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for _ in range(max_len):
            tgt_input_t = cur_tokens.transpose(0, 1)  # (tgt_len, batch)
            tgt_emb = self.embedding(tgt_input_t) * (self.dmodel ** 0.5)

            z_dec_expanded = z_dec.unsqueeze(0)  # (1, batch, dmodel)
            tgt = tgt_emb + z_dec_expanded

            tgt_len = tgt.size(0)
            causal_mask = self._build_causal_mask(tgt_len, device)
            tgt_key_padding_mask = (cur_tokens == self.pad_token_id)  # (batch, tgt_len)

            out = self.decoder(
                tgt,
                memory,
                tgt_mask=causal_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=src_key_padding_mask,
            )  # (tgt_len, batch, dmodel)

            logits = self.output_proj(out[-1])  # last time step: (batch, vocab)
            logits = logits / temperature

            if top_k is not None:
                # top-k sampling
                values, indices = torch.topk(logits, top_k, dim=-1)
                probs = F.softmax(values, dim=-1)
                next_tokens = indices.gather(-1, torch.multinomial(probs, 1))
            else:
                # greedy
                next_tokens = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

            # If EOS, mark finished
            finished = finished | (next_tokens.squeeze(-1) == self.eos_token_id)
            #eos token need in the attributes

            # Append
            cur_tokens = torch.cat([cur_tokens, next_tokens], dim=1)

            # Optional: break if all finished
            if finished.all():
                break

        return cur_tokens  # (batch, generated_len)


    def forward(self, input_ids, attention_mask, target_ids):
        """
        mode = "train"  → teacher-forced autoregressive decode
        mode = "reconstruct" → non-autoregressive (if you keep old path)
        mode = "generate" → use decode_generate externally
        """
        memory, src_key_padding_mask, (c_mu, c_logvar), (s_mu, s_logvar) = \
            self.encode(input_ids, attention_mask)

        z_c = self.reparameterize(c_mu, c_logvar)
        z_s = self.reparameterize(s_mu, s_logvar)

        logits, tgt_output = self.decode_train(
            memory, 
            src_key_padding_mask, 
            z_c, 
            z_s, 
            target_ids
        )

        return logits, tgt_output, (c_mu, c_logvar), (s_mu, s_logvar)


In [23]:
vae_transformer = VAETransformer(d_model, n_heads, n_layers, latent_dim) 

/scratch/404052.1.ood/ipykernel_725746/899198568.py:7: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder =  nn.TransformerEncoder(encoder_layer, nlayers)


In [ ]:
# Reconstruction loss

def reconstruction_loss(logits, tgt_output, pad_token_id):
    # logits: (tgt_len, batch, vocab)
    # tgt_output: (batch, tgt_len)
    logits = logits.transpose(0, 1).contiguous()  # (batch, tgt_len, vocab)
    vocab = logits.size(-1)

    loss = F.cross_entropy(
        logits.view(-1, vocab),
        tgt_output.reshape(-1),
        ignore_index=pad_token_id
    )
    return loss

In [ ]:
# KL Divergence term
def kl_loss(mu, logvar):
    # KL(q(z|x) || N(0, I))
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

def kl_anneal_factor(step, k=0.002, x0=2500):
    # logistic annealing
    return float(1.0 / (1.0 + torch.exp(-k * (step - x0))))


In [ ]:
# HSIC loss
# hilbert schimdt independence criterion

def rbf_kernel(x, sigma=None):
    """
    x: (B, D)
    returns K: (B, B)
    """
    # Pairwise squared distances
    x_norm = (x ** 2).sum(dim=1).view(-1, 1)
    dist = x_norm + x_norm.t() - 2 * (x @ x.t())

    # Median heuristic for sigma
    if sigma is None:
        # Avoid zero distances on diagonal
        dist_no_diag = dist[~torch.eye(dist.size(0), dtype=bool, device=x.device)]
        sigma = torch.median(dist_no_diag)
        sigma = torch.sqrt(0.5 * sigma)

    K = torch.exp(-dist / (2 * sigma ** 2 + 1e-8))
    return K


def hsic(x, y):
    """
    x: (B, Dx)  content latent
    y: (B, Dy)  style latent

    returns scalar HSIC estimate
    """
    B = x.size(0)
    if B < 2:
        return torch.tensor(0.0, device=x.device)

    K = rbf_kernel(x)
    L = rbf_kernel(y)

    # Centering matrix H = I - 1/B * 11^T
    H = torch.eye(B, device=x.device) - (1.0 / B) * torch.ones((B, B), device=x.device)

    # HSIC = (1/(B-1)^2) * trace(KHLH)
    KH = K @ H
    LH = L @ H
    hsic_val = torch.trace(KH @ LH) / ((B - 1) ** 2)

    return hsic_val

In [ ]:
# Contrastive style loss

def contrastive_style_loss(z_style, doc_ids, temperature=0.1):
    z = F.normalize(z_style, dim=-1)
    sim = z @ z.t()  # (B, B)

    mask = torch.eye(z.size(0), device=z.device).bool()
    sim.masked_fill_(mask, -1e9)

    doc_ids = doc_ids.view(-1, 1)
    pos_mask = (doc_ids == doc_ids.t()) & (~mask)

    logits = sim / temperature
    log_probs = F.log_softmax(logits, dim=1)

    pos_log_probs = (log_probs * pos_mask.float()).sum(dim=1) / \
                    (pos_mask.float().sum(dim=1) + 1e-8)

    return -pos_log_probs.mean()

In [ ]:
# Total loss

def total_loss(
    logits,
    tgt_output,
    pad_token_id,
    c_mu, c_logvar,
    s_mu, s_logvar,
    doc_ids,
    step,
    gamma_hsic=1.0,
    lambda_contrast=1.0
):
    recon = reconstruction_loss(logits, tgt_output, pad_token_id)

    kl_c = kl_loss(c_mu, c_logvar)
    kl_s = kl_loss(s_mu, s_logvar)
    kl_total = kl_c + kl_s
    beta = kl_anneal_factor(step)
    kl_term = beta * kl_total

    hsic_term = gamma_hsic * hsic(c_mu, s_mu)

    contrast_term = lambda_contrast * contrastive_style_loss(s_mu, doc_ids)

    loss = recon + kl_term + hsic_term + contrast_term

    return loss, {
        "recon": recon.item(),
        "kl": kl_total.item(),
        "beta": beta,
        "hsic": hsic_term.item(),
        "contrast": contrast_term.item(),
    }
b

In [ ]:
#TODO: Setup dataset

In [ ]:
def train(
    model,
    dataset,
    tokenizer,
    optimizer,
    num_epochs,
    batch_size,
    device,
    gamma_hsic=1.0,
    lambda_contrast=1.0,
):
    model.train()
    model.to(device)

    # We need these
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    global_step = 0

    for epoch in range(num_epochs):
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")

        for batch in pbar:
            texts = batch["text"]
            doc_ids = batch["doc_id"].to(device)

            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                return_tensors="pt"
            )

            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            target_ids = input_ids.clone()

            # Forward pass
            logits, tgt_output, (c_mu, c_logvar), (s_mu, s_logvar) = \
                model(input_ids, attention_mask, target_ids)

            # Compute total loss
            loss, metrics = total_loss(
                logits=logits,
                tgt_output=tgt_output,
                pad_token_id=tokenizer.pad_token_id,
                c_mu=c_mu,
                c_logvar=c_logvar,
                s_mu=s_mu,
                s_logvar=s_logvar,
                doc_ids=doc_ids,
                step=global_step,
                gamma_hsic=gamma_hsic,
                lambda_contrast=lambda_contrast
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            pbar.set_postfix({
                "loss": loss.item(),
                "recon": metrics["recon"],
                "kl": metrics["kl"],
                "beta": metrics["beta"],
                "hsic": metrics["hsic"],
                "contrast": metrics["contrast"],
            })

            global_step += 1

    return model
